# Preprocessing: `team_keepers_adv`  (Equipo — keepers_adv)

Análisis y limpieza individual de esta tabla. Rango: **2018-2025** (temporadas completas).
Salida limpia → `data/processed/team_keepers_adv_clean.csv`.

Estructura estándar: carga → vista general → columnas → cobertura temporal →
duplicados → distribuciones → limpieza específica → export.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

RAW  = Path("../../data/raw/fbref_r")      # ajusta si tu estructura difiere
PROC = Path("../../data/processed")
PROC.mkdir(parents=True, exist_ok=True)

PREFIJO = "laliga"
TABLA   = "team_keepers_adv"
SEASON_MIN, SEASON_MAX = 2018, 2025
CLAVE = ["Season_End_Year", "Squad", "Team_or_Opponent"]

df = pd.read_csv(RAW / f"{PREFIJO}_{TABLA}.csv")
if "Season_End_Year" in df.columns:
    df = df[df["Season_End_Year"].between(SEASON_MIN, SEASON_MAX)].reset_index(drop=True)
print("shape:", df.shape)

## 1. Vista general

In [ ]:
display(df.head(5))
display(df.tail(5))

## 2. Columnas: tipos, nulos, únicos

In [ ]:
resumen = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_nulos": df.isna().sum(),
    "pct_nulos": (df.isna().mean() * 100).round(1),
    "n_unicos": df.nunique(),
}).sort_values("pct_nulos", ascending=False)
display(resumen)

# valores unicos de columnas de baja cardinalidad
for c in df.columns:
    nu = df[c].nunique(dropna=True)
    if nu <= 15:
        print(f"{c} ({nu}):", sorted(df[c].dropna().unique().tolist()))

## 3. Cobertura temporal
Filas por temporada — detecta años incompletos (los datos se congelaron el 18/09/2025;
algunas tablas se actualizaron por última vez en oct-2024).

In [ ]:
if "Season_End_Year" in df.columns:
    por_temp = df["Season_End_Year"].value_counts().sort_index()
    print(por_temp)
    por_temp.plot(kind="bar", figsize=(8,3), title="filas por temporada")
    plt.tight_layout(); plt.show()

## 4. Duplicados por clave natural

In [ ]:
clave = [c for c in CLAVE if c in df.columns]
dup = df.duplicated(subset=clave).sum()
print(f"duplicados exactos por {clave}: {dup}")
if dup:
    display(df[df.duplicated(subset=clave, keep=False)].sort_values(clave).head(10))

## 5. Distribuciones numéricas

In [ ]:
num = df.select_dtypes(include=[np.number]).drop(columns=["Season_End_Year","Born"], errors="ignore")
if len(num.columns):
    n = len(num.columns)
    ncols = 5
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 2.6*nrows))
    for ax, c in zip(np.ravel(axes), num.columns):
        num[c].dropna().hist(bins=30, ax=ax)
        ax.set_title(c, fontsize=8)
    for ax in np.ravel(axes)[n:]:
        ax.axis("off")
    plt.tight_layout(); plt.show()

## 6. Limpieza específica
Pasos estándar + espacio para decisiones propias de esta tabla (documéntalas aquí).

In [ ]:
clean = df.copy()

# columnas numericas disfrazadas de texto -> a numerico
for c in clean.columns:
    if clean[c].dtype == object and c not in ("Squad","Comp","Player","Nation","Pos","Age","Url",
                                              "Home","Away","Date","Day","Time","Round","Venue",
                                              "Referee","Notes","MatchURL","Team_or_Opponent",
                                              "Competition_Name","Gender","Country"):
        conv = pd.to_numeric(clean[c], errors="coerce")
        no_nulos = clean[c].notna().sum()
        if no_nulos and conv.notna().sum() / no_nulos > 0.7:
            clean[c] = conv

# >>> decisiones especificas de esta tabla: anhadelas aqui <<<

print("shape tras limpieza:", clean.shape)
clean.head(3)

## 7. Export a data/processed

In [ ]:
out = PROC / f"{PREFIJO}_{TABLA}_clean.csv"
clean.to_csv(out, index=False)
print("guardado:", out, f"({len(clean)} filas)")